In [ ]:
from ase.io import read
from ase.visualize import view
import nqetools as nqe

import warnings

# Ignore all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

n_beads = 4
timestep = 1.0  # fs
total_steps = 5000
total_steps_md = 100

fix_com = True

stride = 10
temperature = 300
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'ase-mace'

# Expensive settings
driver_args = {'model': 'large',
               'device': 'cuda',
               'default_dtype': 'float64'}

# Cheap settings
driver_args = {'model': 'small',
               'device': 'cuda',
               'default_dtype': 'float32'}

In [ ]:
atoms = nqe.make_dimer(read('fad.xyz', -1))
atoms.center(vacuum=10.0)

clusters = nqe.cluster_atoms(atoms)
print(clusters)
print(len(clusters))


# indexes = nqe.cluster_non_hydrogen_atoms(atoms)
# print(indexes)

view(atoms)

idx_1 = [5,6,7]
idx_2 = [0,1,2]


In [ ]:
f_spt = False

# Plumed hills settings
n_bins = 100
stride_hills = 100
if f_spt:
    plumed_type_opes = "opes-pt1"
    plumed_args_opes = {'idx1': 0,
                        'idx2': 6,
                        'barrier': 0.5,
                        'temperature': temperature,
                        'stride_hills': stride_hills,
                        'explore': True}
    cv_limits = [None, None]
else:
    # Plumed settings
    plumed_type_opes = "opes-diff2"
    plumed_args_opes = {'idx1': 0,
                        'idx2': 4,
                        'idx3': 6,
                        'idx4': 5,
                        'idx5': 9,
                        'idx6': 1,
                        'barrier': 0.01,
                        'temperature': temperature,
                        'stride_hills': stride_hills, }

    plumed_type_opes = "opes-pt2_a"
    plumed_args_opes = {'idx1': 0,
                        'idx2': 6,
                        'idx3': 1,
                        'idx4': 5,
                        'barrier': 0.5,
                        'temperature': temperature,
                        'stride_hills': stride_hills,
                        'explore': True}

    plumed_type_opes = "opes-pt-wob-sep"
    plumed_args_opes = {'idx1': 0,
                        'idx2': 6,
                        'list_1': idx_1,
                        'list_2': idx_2,
                        'barrier': 0.1,
                        'temperature': temperature,
                        'stride_hills': stride_hills,
                        'd_upper': 5.0,
                        'explore': True}
    cv_limits = [[None, None], [None, None]]

In [ ]:
# Run minimization
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          driver_args=driver_args,
                          total_steps=50,
                          )
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    stride=1,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
view(atoms_meta_md)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Load the free energy surface data
fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_md)

# Plot the free energy surface convergence
if f_spt:
    nqe.plot_fes_series_1d(fes_arrays_meta_md, fes_times)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_md, fes_times)

In [ ]:
# Run PIMD OPES metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)

# Plot the free energy surface convergence
fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_pimd)

# Plot the free energy surface convergence
if f_spt:
    nqe.plot_fes_series_1d(fes_arrays_meta_pimd, fes_times)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_pimd, fes_times)

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
if f_spt:
    nqe.plot_fes_series_1d_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])
else:
    nqe.plot_fes_contour_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
if not f_spt:
    nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])